In [1]:
import unittest
import logging
import os
import shutil
import tempfile
import numpy as np
import pandas as pd
import sys
import os

# Add project root to sys.path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from core.execute_porosity_model import execute_porosity_model
from models import get_model as get_pore_model
from meltpool import get_model as get_meltpool_model
import json
from models import get_model as get_pore_model
from download.fetch import getdata

In [2]:
test_params = {
    "Power (W)": 370,
    "Velocity (mm/s)": 1300,
    "Hatch Spacing (um)": 140,
    "Layer Thickness (um)": 30,
    "Spot Size (um)": 100,
    "Preheat Temperature (°C)": 100,
    "Material": "Ti6Al4V",
    "Geometry": "4PB",
}

material_parms = json.load(open('../material_models/test_ti64_rosenthal.json'))

data_source = ['28152209', '26304175']

pore_models = ['lof', 'keyhole', 'gas']

Modified execute porosity model used to get 

In [3]:
def execute_porosity_model(
    itemids,
    data_dir,
    output_dir,
    pore_model_names,
    meltpool_model_name,
    param_dict,
    props,
    totalpores=10000,
    merged_data_dir='data'
):
    # Create dirs if missing
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # Load or download pore data
    poredata_file = os.path.join(merged_data_dir, 'poredf.csv')
    if not os.path.exists(poredata_file):
        print(f"Pore data missing at {poredata_file}. Downloading...")
        getdata(itemids, data_dir, datadir=merged_data_dir)

    poredata = pd.read_csv(poredata_file)

    data_file = os.path.join(merged_data_dir, 'parameters.csv')
    data = pd.read_csv(data_file)

    data = data.rename(columns={'V': 'Velocity (mm/s)', 
                                'P': 'Power (W)', 
                                'Sample': 'Name', 
                                'SS': 'Spot Size (um)',
                                'H': "Hatch Spacing (um)",
                                'L': "Layer Thickness (um)",
                                'T': "Preheat Temperature (°C)",
                                'Geo': 'Geometry',
                                })

    # Melt pool model
    mp_model_class = get_meltpool_model(meltpool_model_name)
    if mp_model_class is None:
        raise ValueError(f"Melt pool model '{meltpool_model_name}' not found.")

    mp_params = {'Model': mp_model_class, 'Properties': props}

    # Instantiate pore models
    selected_models = []
    for pm_name in pore_model_names:
        ModelClass = get_pore_model(pm_name)
        if ModelClass is None:
            raise ValueError(f"Pore model '{pm_name}' not found.")
        model = ModelClass(
            params=param_dict,
            data= data,
            pores=poredata,
            props=mp_params,
            poremodels=selected_models
        )
        selected_models.append(model)
    
    return selected_models

In [4]:
poremodel = execute_porosity_model(data_source, '../data/', 'output/', pore_models, 'Rosenthal', test_params, material_parms, merged_data_dir='../data/')

In [5]:
poremodel[0].sample(N=10)

,Unnamed: 0,Name,Equivalent Spherical Diameter (um),Aspect Ratio,Sphericity,Sample #
0,NaN,CMU02-32-1,15.575914,0.488887,0.701544,1
1,NaN,CMU02-32-1,10.972561,0.463033,0.537061,2
2,NaN,CMU02-32-1,11.382359,0.337320,0.692222,3
3,NaN,CMU02-05-1,8.344561,0.516771,0.688061,4
4,NaN,CMU02-32-1,16.714743,0.510621,0.553457,5
5,NaN,CMU02-32-1,16.301399,0.563513,0.695607,6
6,NaN,CMU02-05-1,15.338387,0.550258,0.699254,7
7,NaN,CMU02-05-1,12.764050,0.462838,0.647473,8
8,NaN,CMU02-05-1,20.675298,0.452383,0.635760,9
9,NaN,CMU02-05-1,12.764050,0.462838,0.647473,10
